# 05 — Interactive MOS notebook

Build, reason about, and query an ontology by writing **Manchester OWL Syntax**
directly in cells. Magics handle the rest.

- `%%mos` — parse cell body as MOS, merge into the active ontology.
- `%reason` — run **HermiT** over the active world (materializes inferences).
- `%%mos_query <relation>` — query the active ontology with an anonymous MOS expression.
- `%mos_show <Name>` — render one entity's axioms back to MOS.
- `%mos_save <path>` — round-trip the active ontology to a `.omn` file.

Tab inside a `%%mos` cell completes keywords, class names, properties, and individuals.

In [1]:
%load_ext omny.jupyter

## Build a small pizza ontology

In [2]:
%%mos
ObjectProperty: hasTopping
Class: Pizza
Class: Cheese
Class: Tomato
Class: Margherita
    SubClassOf: Pizza
    EquivalentTo: Pizza and (hasTopping some Cheese)

## Reason

HermiT runs over the asserted graph and materializes inferences back into the world.

In [3]:
%reason

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/site-packages/owlready2/hermit:/usr/local/lib/python3.12/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpd5tu84ci


* Owlready2 * HermiT took 0.6861345767974854 seconds
* Owlready * Reparenting notebook.Margherita: {owl.Thing, notebook.Pizza} => {notebook.Pizza}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## Query

Ask: which named class is equivalent to `Pizza and (hasTopping some Cheese)`?

In [4]:
%%mos_query equiv
Pizza and (hasTopping some Cheese)

http://omny.test/notebook#Margherita


## Render one class back to MOS

In [5]:
%mos_show Margherita

Class: :Margherita
    SubClassOf: :Pizza
    EquivalentTo: :Pizza and :hasTopping some :Cheese



## Extend the ontology

Manchester cells merge incrementally — re-running `%%mos` adds, it doesn't replace.

In [6]:
%%mos
Class: Vegetarian
    EquivalentTo: Pizza and not (hasTopping some Meat)
Class: Meat

Re-run reasoning to pick up the new axioms, then query subclasses of Pizza.

In [7]:
%reason

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/site-packages/owlready2/hermit:/usr/local/lib/python3.12/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpm6cbhc19


* Owlready2 * HermiT took 0.46922874450683594 seconds
* Owlready * Reparenting notebook.Vegetarian: {owl.Thing} => {notebook.Pizza}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [8]:
%%mos_query sub
Pizza

http://omny.test/notebook#Margherita
http://omny.test/notebook#Vegetarian


## Save the whole ontology

In [9]:
%mos_save /tmp/pizza-built.omn

wrote /tmp/pizza-built.omn


## Takeaway

You wrote MOS, asked HermiT to reason, queried via SPARQL, and round-tripped to
disk — without leaving Manchester syntax. The live ontology is also exposed in
the user namespace as `mos_onto` (and the world as `mos_world`) for any Python
follow-up you want.

In [10]:
print("classes:", [c.name for c in mos_onto.classes()])
print("properties:", [p.name for p in mos_onto.object_properties()])

classes: ['Pizza', 'Cheese', 'Tomato', 'Margherita', 'Vegetarian', 'Meat']
properties: ['hasTopping']
